<a href="https://colab.research.google.com/github/SunJ2432/ai-image-compliance/blob/main/notebooks/03_descriptor_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A3 — Image description + subclassification
Qwen3.5-4B (vision) and Qwen2.5-1.5B-Instruct (classification). Run on a Colab T4. Models are loaded **sequentially** to reduce VRAM pressure.

In [1]:
%pip install -q -U "transformers>=4.57" accelerate safetensors pillow pandas pytest timm

If dependencies changed, restart the Colab runtime before continuing. Do not install `torchao` for this workflow.

In [2]:
from pathlib import Path
import sys
GITHUB_REPO = "SunJ2432/ai-image-compliance"
repo_dir = Path("/content") / GITHUB_REPO.split("/")[-1]
if not repo_dir.exists():
    !git clone https://github.com/{GITHUB_REPO}.git {repo_dir}
else:
    !git -C {repo_dir} pull --ff-only
assert (repo_dir / "samples_tiny/manifest.csv").is_file(), "Dataset missing; check GitHub sync"
if str(repo_dir / "src") not in sys.path:
    sys.path.insert(0, str(repo_dir / "src"))
print("Repo:", repo_dir)

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 6 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.20 KiB | 613.00 KiB/s, done.
From https://github.com/SunJ2432/ai-image-compliance
   8a522e3..8c447b5  main       -> origin/main
Updating 8a522e3..8c447b5
Fast-forward
 src/imgcompliance/captioner/model.py | 7 ++++---
 1 file changed, 4 insertions(+), 3 deletions(-)
Repo: /content/ai-image-compliance


In [3]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Select T4 GPU under Runtime > Change runtime type"

GPU: Tesla T4


## Stage 1 — Caption all images. The Qwen3.5-4B model downloads on first call. The identity-safety instruction is in `captioner/model.py`.

In [4]:
import pandas as pd
from imgcompliance.captioner import Captioner
manifest = repo_dir / "samples_tiny/manifest.csv"
df = pd.read_csv(manifest)
captioner = Captioner()
caption_rows = []
for _, row in df.iterrows():
    path = manifest.parent / row["filename"]
    assert path.exists(), f"Missing image: {path}"
    caption = captioner.describe(path)
    caption_rows.append({"filename": row["filename"], "ground_truth": row["label"], "caption": caption})
    print(path.name, caption["scene_summary"][:120])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


real_001.jpg A mountain landscape featuring a prominent stratified peak with snow patches, surrounded by green meadows and scattered 
ai_001.jpg A mountain landscape featuring a steep, rocky peak with patches of snow, surrounded by green alpine meadows dotted with 
real_002.jpg A landscape view of a densely forested mountain range under an overcast sky, with lush green vegetation in the foregroun
ai_002.jpg A landscape view of lush green hills and dense vegetation under an overcast, gray sky with mist or low clouds obscuring 
real_003.jpg A landscape photograph showing rolling white sand dunes in the foreground with sparse dry vegetation, leading to a valle
ai_003.jpg A desert landscape featuring rolling sand dunes in the foreground with a distant mountain range under a sky with scatter


## Release the VLM before loading the classifier

In [10]:
print("captioner exists:", "captioner" in globals())
print("caption_rows exists:", "caption_rows" in globals())
print("caption_rows count:", len(caption_rows) if "caption_rows" in globals() else 0)

captioner exists: False
caption_rows exists: True
caption_rows count: 6


In [11]:
import gc
import torch

if "captioner" in globals():
    del captioner

gc.collect()
torch.cuda.empty_cache()

print("Captioning finished; GPU cache cleared")

Captioning finished; GPU cache cleared


## Stage 2 — Restore your A2 checkpoint and classify. If checkpoint is missing after a runtime reset, restore it or rerun A2 training first. The default context does **not** assert that any depicted entity has been verified.

In [12]:
%pip install -U qwen-vl-utils

In [13]:
from imgcompliance.detector import Detector
from imgcompliance.classifier import Classifier
checkpoint = Path("/content/detector_head.pt")
assert checkpoint.exists(), "Restore your A2 checkpoint to /content/detector_head.pt"
detector = Detector(checkpoint=checkpoint)
classifier = Classifier()
results = []
for item in caption_rows:
    path = manifest.parent / item["filename"]
    detection = detector.predict(path)
    context = {"declared_purpose": None, "verified_real_entity": False}
    sub = classifier.classify(item["caption"], detection, context)
    results.append({
        "filename": item["filename"], "ground_truth": item["ground_truth"],
        "detected": detection["label"], "ai_score": round(detection["prob"], 4),
        "scene_summary": item["caption"]["scene_summary"],
        "depicts_people": item["caption"]["depicts_people"],
        "appears_photorealistic": item["caption"]["appears_photorealistic"],
        "possible_real_world_reference": item["caption"]["possible_real_world_reference"],
        "caption_format": item["caption"]["format"],
        "subclass": sub["subclass"], "rationale": sub["rationale"]
    })
results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv("/content/a3_results.csv", index=False)
print("Saved /content/a3_results.csv")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

,filename,ground_truth,detected,ai_score,scene_summary,depicts_people,appears_photorealistic,possible_real_world_reference,caption_format,subclass,rationale
0,real/real_001.jpg,real,real,0.4396,A mountain landscape featuring a prominent str...,False,True,None,json,not_ai_generated,Detector classified the image as real; not ind...
1,ai_generated/ai_001.jpg,ai_edited,ai,0.5855,"A mountain landscape featuring a steep, rocky ...",False,True,None,json,general_synthetic,The image depicts a mountain landscape but lac...
2,real/real_002.jpg,real,real,0.4880,A landscape view of a densely forested mountai...,False,True,None,json,not_ai_generated,Detector classified the image as real; not ind...
3,ai_generated/ai_002.jpg,ai_edited,ai,0.6495,A landscape view of lush green hills and dense...,False,True,None,json,general_synthetic,The image depicts a realistic landscape but la...
4,real/real_003.jpg,real,real,0.3546,A landscape photograph showing rolling white s...,False,True,None,json,not_ai_generated,Detector classified the image as real; not ind...
5,ai_generated/ai_003.jpg,ai_edited,real,0.4683,A desert landscape featuring rolling sand dune...,False,True,None,json,not_ai_generated,Detector classified the image as real; not ind...


Saved /content/a3_results.csv


## Unit tests (no model downloads)

In [9]:
!PYTHONPATH={repo_dir}/src python -m pytest -q {repo_dir}/tests/test_understand.py

.............                                                            [100%]
13 passed in 0.04s


**Interpretation:** This is a smoke test using the same six images as A2 training. It does not establish real-world accuracy. A deepfake subclass needs external evidence of a real-world entity; the landscape samples should generally be `general_synthetic` when the detector says `ai`.